# Numerical Integration

In numerical integration, we want to **calculate the area under the curve to find the definite integral** (an integral with defined limits) of a function that cannot be solved analytically. 
As you have learned in previous years, given a function $f(x)$, its definite integral between $a$ and $b$ is the area bounded between the function itself and the x-axis.  

In most cases, we do not have an antiderivative that can be calculated using "regular" differentiation. Therefore, as always, we prefer to replace the function $f(x)$ with a polynomial $P(x)$ that is very similar to it (as close as we desire) within the requested domain.  In integration methods, as in all the numerical methods we have learned so far, what distinguishes them is the information we have at our disposal and the required level of accuracy.


Today, we're gonna go over three methods:

Trapezodial rule - $$A = \frac{1}{2}(x_{i+1} - x_i)[f(x_i) + f(x_{i+1})]$$

The idea is to draw a trapezoid to estimate the area: the trapezoidal method is equivalent to a first-order polynomial approximation of a given function.  

Simpson's 1/3 rule - 
$$\frac{1}{3}h \left[ f(x_0) + 4 \sum_{\text{odd } i=1}^{n-1} f(x_i) + 2 \sum_{\text{even } i=2}^{n-2} f(x_i) + f(x_n) \right]$$

This method is essentially a second-order method. We add a midpoint between $a$ and $b$. It must be performed with an even number of segments (an odd number of points).


Richardson extrapolation (will be used in Rumberg) - 

$$I_{j,k} = \frac{4^k I_{j+1,k-1} - I_{j,k-1}}{4^k - 1}$$

$$k = 1, 2, 3, \dots \quad j = 0, 1, 2, \dots$$

Richardson extrapolation essentially combines two imperfect integral estimates from the trapezoidal rule, which cancel each other out and thereby reduce the error component. We enter these into the Romberg table that you learned about in the lecture.


### Trapezoid Rule

In [2]:
import numpy as np

# --- Setup block: Defining a super simple examle ---
def f(x):
    return x**2

a = 0.0
b = 3.0
num_segments = 3  # Division into exactly 3 equal segments

# Generating easy-to-read node coordinates: [0., 1., 2., 3.]
x_nodes = np.linspace(a, b, num_segments + 1)
y_nodes = f(x_nodes)  # Resulting Y values: [0., 1., 4., 9.]

# --- The execution block ---
h = (b - a) / num_segments  # h = 1.0

# Composite Trapezoid formula: (h/2) * [ y_0 + 2*(y_1 + y_2) + y_3 ]
# Task for students: Match the code slice with the numbers: y_nodes[1:-1] is [1., 4.]
trapezoid_result = (h / 2.0) * (y_nodes[0] + 2.0 * np.sum(y_nodes[1:-1]) + y_nodes[-1])

print(f"Nodes X: {x_nodes}")
print(f"Nodes Y: {y_nodes}")
print("-" * 30)
print(f"Trapezoid Estimate: {trapezoid_result}")

Nodes X: [0. 1. 2. 3.]
Nodes Y: [0. 1. 4. 9.]
------------------------------
Trapezoid Estimate: 9.5


### Simpson's 1/3 Rule

In [3]:
# Critical Requirement: The number of segments (num_segments) must be even!

h = (b - a) / num_segments

# Summing the nodes at odd indices (multiplied by 4 in the formula)
# Starts at index 1, goes up to the last internal node, with a step size of 2
sum_odd = np.sum(y_nodes[1:-1:2])

# Summing the nodes at even internal indices (multiplied by 2 in the formula)
# Starts at index 2, goes up to the last internal node, with a step size of 2
sum_even = np.sum(y_nodes[2:-1:2])

# Final Simpson's 1/3 calculation
simpson_result = (h / 3.0) * (y_nodes[0] + 4.0 * sum_odd + 2.0 * sum_even + y_nodes[-1])

print(f"Simpson's 1/3 Estimate: {simpson_result}")

Simpson's 1/3 Estimate: 7.0


In [5]:
# The first column (index 0) has already been filled using the Trapezoid Rule for j=0 and j=1
R = [
    [8.5],      # Row j=0: Trapezoid estimate for 1 segment (full step size h)
    [8.15]      # Row j=1: Trapezoid estimate for 2 segments (half step size h/2)
]

# We want to compute the first improved estimate: Row j=1, Column k=1
j = 1
k = 1

# 1. Extract the "dense grid" estimate (located in the current row, previous column)
I_dense = R[j][k - 1] 

# 2. Extract the "sparse grid" estimate (located in the previous row, previous column)
I_sparse = R[j - 1][k - 1]

# 3. Apply the universal Richardson Extrapolation formula
numerator = (4 ** k) * I_dense - I_sparse
denominator = (4 ** k) - 1
I_improved = numerator / denominator

# Append the newly computed improved estimate to the current row
R[j].append(I_improved)

print(f"The updated row j=1 is now: {R[1]}")
print(f"Richardson value (Simpson's 1/3 equivalent): {R[1][1]}")

The updated row j=1 is now: [8.15, 8.033333333333333]
Richardson value (Simpson's 1/3 equivalent): 8.033333333333333
